In [1]:
# ============================================================
# CELL 0: IMPORTS, GLOBAL CONFIGURATION, AND DATA LOADING
# ============================================================
#
# Perturbation-Based Conformal Prediction for Neural Operators
# 2D Navier-Stokes  |  navierstokes-v1e-5-n1200-t20
#
# ============================================================
#
# PURPOSE OF THIS NOTEBOOK:
# -------------------------
# This notebook compares different Uncertainty Quantification (UQ) methods
# for a Fourier Neural Operator (FNO) applied to the 2D Navier-Stokes equation,
# with a focus on the DATA-SCARCE regime.
#
# All methods are wrapped in a split conformal prediction framework to
# produce prediction bands with finite-sample simultaneous coverage
# guarantees. The core question: which uncertainty proxy produces the
# NARROWEST bands while maintaining the target coverage?
#
# METHODS COMPARED:
# -----------------
#   1. Perturbation (ours): Train two FNOs — one on original labels and one
#      on slightly perturbed labels — and use pointwise disagreement as the
#      uncertainty scale. No extra data needed.
#   2. MC Dropout: Keep dropout active at test time and estimate predictive
#      variance from 20 stochastic forward passes.
#   3. Laplace Approximation: Diagonal last-layer Laplace approximation
#      provides a scalar predictive variance per sample.
#   4. UQNO (Ma et al. arXiv:2402.01960): Train a separate quantile FNO
#      (E-network) to predict error bounds. Must split the training data
#      between the base FNO and the E-network.
#   5. Unscaled (σ≡1): Constant-width conformal bands (ablation baseline).
#
# DATA BUDGET (Equal total: 1200 samples):
# -----------------------------------------
# Our split:
#   a_tr_ours (800): base FNO training  [used by Perturbation, Laplace, MCDropout]
#   a_cal     (200): conformal calibration  [shared across all methods]
#   a_test    (200): evaluation             [shared across all methods]
#
# Ma's split:
#   a_tr1_ma  (400): base FNO training  [used by UQNO]
#   a_tr2_ma  (400): quantile FNO (E) training  [required by UQNO]
#   a_cal     (200): conformal calibration  [shared]
#   a_test    (200): evaluation             [shared]
#
# REVIEWER-REQUESTED ADDITIONS:
# ------------------------------
# This version adds the following experiments beyond the original submission:
#   A. ± Standard errors for coverage and bandwidth (1000 reshuffles)
#   B. Ablation study: effect of spatial smoothing (K) and floor (τ₀)
#   C. Perturbation noise level (σ_ε) sensitivity
#   D. Random seed sensitivity (retrain base + perturbed FNO)
#
# ============================================================

# ===================== IMPORTS =====================
import os          # Operating system interface (file paths, environment)
import math        # Basic math functions (ceil, floor, etc.)
import warnings    # Control warning messages display

import numpy as np    # NumPy: fundamental package for array/matrix computation
import scipy.io       # SciPy: loading MATLAB .mat data files
import torch          # PyTorch: the deep learning framework
import torch.nn as nn             # Neural network modules (layers, loss functions)
import torch.nn.functional as F   # Functional API (activations, pooling, etc.)

# PyTorch functional transforms for advanced differentiation:
# jvp: Jacobian-vector product; vmap: vectorized map; functional_call: call module with params
from torch.func import jvp, vmap, functional_call

# DataLoader: creates mini-batches; TensorDataset: wraps tensors into a dataset object
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt        # Plotting library
import matplotlib.gridspec as gridspec  # Fine-grained subplot layout control
import h5py                             # Reads HDF5 files (a binary data format)

# ===================== GLOBAL CONFIGURATION =====================

# Suppress all warnings to keep output clean
warnings.filterwarnings("ignore")

# Set random seeds for REPRODUCIBILITY — ensures identical results each run
torch.manual_seed(42)
np.random.seed(42)

# Use GPU if available (much faster for neural networks), otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Remove leftover checkpoint files from previous runs
for ckpt in ["fno_scarce.pt", "fno_ma_scarce.pt"]:
    if os.path.exists(ckpt):
        os.remove(ckpt)

# Configure matplotlib for clean, publication-quality plots
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 12, "axes.labelsize": 11,
    "legend.fontsize": 10, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150,
})

# ===================== VISUAL CONFIGURATION =====================
# Consistent colors, markers, and labels for each UQ method across all figures

COLORS  = {"MCDropout": "#2166AC",      # Blue
           "Laplace": "#D6604D",        # Red-orange
           "UQNO": "#4DAC26",           # Green
           "Perturbation": "#9B59B6",   # Purple (our method)
           "NoSigma": "#888888"}        # Gray (ablation baseline)

MARKERS = {"MCDropout": "^",            # Triangle-up
           "Laplace": "*",              # Star
           "UQNO": "s",                 # Square
           "Perturbation": "P",         # Plus (filled)
           "NoSigma": "o"}             # Circle

LABELS  = {"MCDropout": "MCDropout", "Laplace": "Laplace",
           "UQNO": "UQNO", "Perturbation": "Perturbation(ours)",
           "NoSigma": "Unscaled ($\\sigma\\equiv 1$)"}

METHODS = list(COLORS.keys())  # All 5 methods

# ============================================================
# 1. DATA LOADING
# ============================================================
# The dataset contains solutions to the 2D Navier-Stokes equations
# (governing fluid flow / vorticity).
#
# Each sample is a time series of 2D fields on a 64x64 spatial grid.
# Shape: (N, 64, 64, T) where N=samples, 64x64=spatial, T=20 time steps.
#
# We split time into:
#   T_in  = 10: input  (the "initial condition" given to the FNO)
#   T_out = 10: output (what the FNO must predict)
#
# The FNO learns the solution operator: u(x,y, t=0..9) --> u(x,y, t=10..19)
# ============================================================

DATA_PATH = "/kaggle/input/datasets/erikdeng/navierstokes-v1e-5-n1200-t20"

def load_ns_data(path=DATA_PATH, T_in=10, T_out=10):
    """
    Load 2D Navier-Stokes vorticity field data from .mat files.

    Parameters:
        path  : directory containing .mat data files
        T_in  : number of input time steps (initial condition)
        T_out : number of output time steps (prediction target)

    Returns:
        a     : input tensor  (N, 64, 64, T_in)
        b     : target tensor (N, 64, 64, T_out)
        S     : spatial resolution (64)
        T_in, T_out : number of input/output time steps
    """
    files = sorted([f for f in os.listdir(path) if f.endswith('.mat')])
    print("Files:", files)
    fname = os.path.join(path, files[0])

    # Try HDF5 format first (MATLAB v7.3+ uses HDF5 internally)
    try:
        with h5py.File(fname, 'r') as f:
            raw_u = np.array(f['u'], dtype=np.float32)
        # HDF5/MATLAB stores dimensions in transposed order → permute to (N,H,W,T)
        u = torch.FloatTensor(raw_u).permute(3, 1, 2, 0)
    except:
        # Fallback: older MATLAB format via scipy
        data = scipy.io.loadmat(fname)
        raw_u = data['u'].astype(np.float32)
        u = torch.FloatTensor(raw_u)
        if u.shape[-1] == 1200 or u.shape[-1] == 5000:
            u = u.permute(3, 0, 1, 2)  # → (N, 64, 64, T)

    N, S1, S2, T = u.shape
    print(f"u tensor: N={N}  S={S1}  T={T}  shape={u.shape}")

    # Split time axis: first T_in steps = input, next T_out steps = target
    a = u[..., :T_in]               # (N, 64, 64, T_in)
    b = u[..., T_in:T_in + T_out]   # (N, 64, 64, T_out)
    return a, b, S1, T_in, T_out

# Load all 1200 samples
a_all, b_all, S, T_in, T_out = load_ns_data(T_in=10, T_out=10)
a_all_orig, b_all_orig = a_all.clone(), b_all.clone()  # Keep originals before shuffle

# ===================== DATA-SCARCE SPLIT =====================
# Randomly shuffle and split the 1200 samples
perm = torch.randperm(1200)
a_all = a_all_orig[perm]
b_all = b_all_orig[perm]

N_TR_OURS = 800   # Our base FNO training set (larger — no E-network needed)
N_TR1_MA  = 400   # Ma's base FNO (smaller — must share budget with E-network)
N_TR2_MA  = 400   # Ma's quantile FNO (E-network) training set
N_CAL     = 200   # Calibration set for conformal prediction (shared)
N_TEST    = 200   # Test set for evaluation (shared)

# Our split: [0..799]=train, [800..999]=cal, [1000..1199]=test
a_tr_ours = a_all[:N_TR_OURS];                          b_tr_ours = b_all[:N_TR_OURS]
# Ma's split: [0..399]=base_train, [400..799]=E_train
a_tr1_ma  = a_all[:N_TR1_MA];                           b_tr1_ma  = b_all[:N_TR1_MA]
a_tr2_ma  = a_all[N_TR1_MA:N_TR1_MA+N_TR2_MA];         b_tr2_ma  = b_all[N_TR1_MA:N_TR1_MA+N_TR2_MA]
# Shared calibration and test sets
a_cal     = a_all[N_TR_OURS:N_TR_OURS+N_CAL];          b_cal     = b_all[N_TR_OURS:N_TR_OURS+N_CAL]
a_test    = a_all[N_TR_OURS+N_CAL:N_TR_OURS+N_CAL+N_TEST]; b_test = b_all[N_TR_OURS+N_CAL:N_TR_OURS+N_CAL+N_TEST]

print(f"\nData-scarce split (equal budget = {N_TR_OURS + N_CAL + N_TEST} samples):")
print(f"  Our base FNO    : {len(a_tr_ours)} samples")
print(f"  Ma base FNO     : {len(a_tr1_ma)} samples")
print(f"  Ma quantile FNO : {len(a_tr2_ma)} samples")
print(f"  Calibration     : {len(a_cal)} samples")
print(f"  Test            : {len(a_test)} samples")

# Shorter aliases used later
a_tr1 = a_tr_ours; b_tr1 = b_tr_ours
a_tr2 = a_tr2_ma;  b_tr2 = b_tr2_ma

Device: cuda
Files: ['NavierStokes_V1e-5_N1200_T20.mat']
u tensor: N=1200  S=64  T=20  shape=torch.Size([1200, 64, 64, 20])

Data-scarce split (equal budget = 1200 samples):
  Our base FNO    : 800 samples
  Ma base FNO     : 400 samples
  Ma quantile FNO : 400 samples
  Calibration     : 200 samples
  Test            : 200 samples


In [2]:
# ============================================================
# CELL 1: FNO-2D ARCHITECTURE
# ============================================================
# The Fourier Neural Operator (FNO) learns mappings between function spaces.
#
# Key idea: instead of local convolutions (CNNs), FNO performs
# convolution in FOURIER (frequency) SPACE, capturing global patterns
# in a single layer.
#
# Architecture:
#   1. Lift input to high-dimensional space  (fc0: T_in+2 → width)
#   2. Apply FNO blocks  (spectral conv + 1x1 skip + GELU), repeated `depth` times
#   3. Project back to output  (fc1: width → 128 → fc2: 128 → T_out)
#
# Reference: Li et al. "Fourier Neural Operator for Parametric PDEs" (ICLR 2021)
# ============================================================

class SpectralConv2d(nn.Module):
    """
    2D Spectral Convolution — the core building block of FNO.

    Performs convolution in Fourier domain:
      1. FFT the input to frequency space
      2. Multiply low-frequency modes by learnable complex weights
      3. Zero-pad high-frequency modes (effectively a low-pass filter)
      4. Inverse FFT back to physical space

    This is equivalent to a GLOBAL convolution in physical space,
    capturing long-range spatial dependencies in one layer.

    Parameters:
        ic : input channels
        oc : output channels
        m1 : Fourier modes kept along spatial dim 1
        m2 : Fourier modes kept along spatial dim 2
    """
    def __init__(self, ic, oc, m1, m2):
        super().__init__()
        sc = 1 / (ic * oc)           # Xavier-like initialization scale
        self.m1, self.m2 = m1, m2     # Number of Fourier modes to retain
        # Complex-valued learnable weights for low-frequency Fourier coefficients.
        # W1 = weights for positive frequencies (top rows of FFT output)
        # W2 = weights for negative frequencies (bottom rows of FFT output)
        self.W1 = nn.Parameter(sc * torch.randn(ic, oc, m1, m2, dtype=torch.cfloat))
        self.W2 = nn.Parameter(sc * torch.randn(ic, oc, m1, m2, dtype=torch.cfloat))

    def forward(self, x):
        """
        x shape: (Batch, Channels, Height, Width)
        returns:  (Batch, out_channels, Height, Width)
        """
        B, C, H, W = x.shape

        # Step 1: 2D Real FFT → complex coefficients, shape (B, C, H, W//2+1)
        # rfft2 exploits conjugate symmetry of real signals → only half the freqs
        xf = torch.fft.rfft2(x)

        # Step 2a: Multiply FIRST m1 rows × first m2 cols (positive freqs) by W1
        # einsum contracts over input channels 'i', producing output channels 'o'
        out1 = torch.einsum("bihw,iohw->bohw", xf[:, :, :self.m1,  :self.m2], self.W1)

        # Step 2b: Multiply LAST m1 rows × first m2 cols (negative freqs) by W2
        out2 = torch.einsum("bihw,iohw->bohw", xf[:, :, -self.m1:, :self.m2], self.W2)

        # Step 3: Reconstruct full frequency tensor by zero-padding high frequencies
        pad_Y = W // 2 + 1 - self.m2                                    # padding along freq dim 2
        mid_X = H - 2 * self.m1                                          # middle rows = high freq = zeros
        mid_z = xf.new_zeros(B, self.W1.shape[1], mid_X, W // 2 + 1)    # zero block for high freqs
        out = torch.cat([F.pad(out1, (0, pad_Y)),   # top corner (low freq) + right zero-pad
                         mid_z,                       # middle = zeros (high freq filtered out)
                         F.pad(out2, (0, pad_Y))],    # bottom corner (neg freq) + right zero-pad
                        dim=2)

        # Step 4: Inverse FFT → back to physical (spatial) domain
        return torch.fft.irfft2(out, s=(H, W))


class FNOBlock(nn.Module):
    """
    One FNO block = SpectralConv + 1x1 skip connection + GELU activation.

    Output: GELU( SpectralConv(x) + Conv1x1(x) )

    The spectral conv captures global patterns; the 1x1 conv captures local patterns.
    """
    def __init__(self, w, m1, m2):
        super().__init__()
        self.conv = SpectralConv2d(w, w, m1, m2)  # Global: Fourier-space conv
        self.skip = nn.Conv2d(w, w, 1)             # Local: pointwise 1x1 conv

    def forward(self, x):
        return F.gelu(self.conv(x) + self.skip(x))


class FNO2d(nn.Module):
    """
    Complete 2D Fourier Neural Operator.

    Input:  (B, 64, 64, T_in)  — vorticity at T_in time steps
    Output: (B, 64, 64, T_out) — predicted vorticity at T_out future steps

    Architecture:
      fc0:    Linear(T_in+2 → width)       # "+2" = 2D grid coordinates appended
      blocks: [FNOBlock] × depth            # Stacked spectral conv blocks
      dropout: Dropout(p=drop)              # Regularization (randomly zero neurons)
      fc1:    Linear(width → 128)  + GELU   # First projection layer
      fc2:    Linear(128 → T_out)           # Final output layer
    """
    def __init__(self, modes=12, width=32, T_in=10, T_out=10, depth=4, drop=0.1):
        super().__init__()
        self.T_out  = T_out
        self.fc0    = nn.Linear(T_in + 2, width)   # Lifting: (T_in+2) → width channels
        self.blocks = nn.ModuleList([FNOBlock(width, modes, modes) for _ in range(depth)])
        self.dropout = nn.Dropout(drop)  # During training, randomly zero 10% of activations
        self.fc1    = nn.Linear(width, 128)
        self.fc2    = nn.Linear(128, T_out)

    def _grid(self, B, Sp, dev):
        """
        Create 2D positional grid in [0,1]x[0,1].
        Appended to input so the network knows each pixel's spatial position.
        Returns shape (B, Sp, Sp, 2).
        """
        x = torch.linspace(0, 1, Sp, device=dev)
        gx, gy = torch.meshgrid(x, x, indexing='ij')
        return torch.stack([gx, gy], -1).unsqueeze(0).expand(B, -1, -1, -1)

    def forward_with_features(self, a):
        """
        Forward pass that ALSO returns intermediate feature vectors.

        Features = spatially-averaged activations after each FNO block.
        Used by the Laplace approximation for uncertainty estimation.

        Returns:
            x    : (B, 64, 64, T_out) — predicted vorticity
            feat : (B, depth*width)   — concatenated feature vectors (e.g., B×128)
        """
        B, Sp, _, Ti = a.shape
        g   = self._grid(B, Sp, a.device)                 # (B, 64, 64, 2)
        x   = self.fc0(torch.cat([a, g], -1))              # (B, 64, 64, width)
        x   = x.permute(0, 3, 1, 2)                        # (B, width, 64, 64) for conv2d
        fs  = []
        for blk in self.blocks:
            x = blk(x)
            # Global average pooling: collapse spatial dims → one vector per sample
            fs.append(x.mean(dim=[-2, -1]))                 # each (B, width)
        feat = torch.cat(fs, -1)                            # (B, depth*width) = (B, 128)
        x   = self.dropout(x)
        x   = F.gelu(self.fc1(x.permute(0, 2, 3, 1)))      # (B, 64, 64, 128)
        x   = self.fc2(x)                                   # (B, 64, 64, T_out)
        return x, feat

    def forward(self, a):
        """Standard forward pass — discards features."""
        p, _ = self.forward_with_features(a); return p


def rel_l2(p, t):
    """
    Relative L2 error: ||pred - true||_2 / ||true||_2, averaged over batch.
    A value of 0.01 means ~1% prediction error relative to the target magnitude.
    """
    diff  = (p - t).reshape(p.shape[0], -1)
    denom = t.reshape(t.shape[0], -1).norm(dim=1) + 1e-8
    return (diff.norm(dim=1) / denom).mean()

In [3]:
# ============================================================
# CELL 2: TRAIN BASE FNO (OUR MODEL — 800 SAMPLES)
# ============================================================
# This is the primary predictive model used by Perturbation, Laplace,
# MC Dropout, and the Unscaled baseline. It sees all 800 training
# trajectories, giving it the maximum possible data budget.
# ============================================================

def train_fno(model, a_tr, b_tr, a_val, b_val, epochs=500,
              bs=10, lr=1e-3, ckpt="fno_scarce.pt"):
    """
    Train an FNO with relative L2 loss.

    Uses Adam optimizer with step learning rate decay (halve lr every 200 epochs).
    Saves the best model checkpoint (lowest training loss).

    Parameters:
        model  : FNO2d to train
        a_tr   : training inputs  (N_tr, 64, 64, T_in)
        b_tr   : training targets (N_tr, 64, 64, T_out)
        a_val  : validation inputs  (for monitoring only)
        b_val  : validation targets
        epochs : number of full passes through the training data
        bs     : mini-batch size (small=10 because dataset is small)
        lr     : initial learning rate
        ckpt   : filename to save best model weights
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)        # Adaptive lr optimizer
    sch = torch.optim.lr_scheduler.StepLR(opt, 200, 0.5)     # Halve lr every 200 epochs
    dl  = DataLoader(TensorDataset(a_tr, b_tr), min(bs, len(a_tr)), shuffle=True)
    model.to(DEVICE); best = float('inf')
    for ep in range(epochs):
        model.train(); tot = 0                                # Enable dropout for training
        for a, b in dl:
            a, b = a.to(DEVICE), b.to(DEVICE)
            loss = rel_l2(model(a), b)                        # Forward: compute relative L2 loss
            opt.zero_grad(); loss.backward(); opt.step()      # Backward: compute gradients & update
            tot += loss.item()
        sch.step(); avg = tot / len(dl)
        if ep % 50 == 0:                                      # Print progress every 50 epochs
            model.eval()
            with torch.no_grad():
                vl = rel_l2(model(a_val.to(DEVICE)).cpu(), b_val).item()
            print(f"  ep{ep:3d}  train={avg:.4f}  val={vl:.4f}")
        if avg < best:
            best = avg; torch.save(model.state_dict(), ckpt)  # Save best checkpoint
    print(f"Best train rel-L2 = {best:.4f}")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))  # Load best weights
    return model


# ----- Train OUR base FNO (800 training samples) -----
print(f"\nTraining base FNO (ours) on {N_TR_OURS} samples...")
model = FNO2d(modes=12, width=32, T_in=T_in, T_out=T_out, depth=4, drop=0.1)
model = train_fno(model, a_tr_ours, b_tr_ours, a_cal, b_cal,
                  epochs=500, ckpt="fno_scarce.pt")
model.eval()  # Permanently switch to eval mode (disable dropout)


Training base FNO (ours) on 800 samples...
  ep  0  train=0.6760  val=0.4548
  ep 50  train=0.1875  val=0.2495
  ep100  train=0.1601  val=0.2477
  ep150  train=0.1483  val=0.2533
  ep200  train=0.1387  val=0.2533
  ep250  train=0.1347  val=0.2592
  ep300  train=0.1320  val=0.2597
  ep350  train=0.1297  val=0.2601
  ep400  train=0.1266  val=0.2631
  ep450  train=0.1253  val=0.2634
Best train rel-L2 = 0.1244


FNO2d(
  (fc0): Linear(in_features=12, out_features=32, bias=True)
  (blocks): ModuleList(
    (0-3): 4 x FNOBlock(
      (conv): SpectralConv2d()
      (skip): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc1): Linear(in_features=32, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [4]:
# ============================================================
# CELL 3: TRAIN PERTURBED FNO
# ============================================================
# Perturbation method:
#   1. Train base FNO on original targets: f(a) → b
#   2. Add small Gaussian noise to targets: b' = b + ε,  ε ~ N(0, σ_ε²)
#   3. Train second FNO on perturbed targets: f_pert(a) → b'
#   4. Uncertainty at test time: σ(a) = |f(a) - f_pert(a)|
#
# Intuition: Where the learned mapping is stable (well-constrained by data),
# a tiny label perturbation won't change predictions → small σ.
# Where the mapping is unstable (under-constrained), predictions diverge → large σ.
#
# Crucially, both FNOs use the FULL training set — no data is reserved for a
# separate uncertainty model. This is the key advantage over UQNO in the
# data-scarce regime.
#
# The perturbation magnitude σ_ε = 0.05 × std(u_train) is the default.
# Section B below sweeps over different noise levels as requested by reviewers.
# ============================================================

# Perturbation magnitude = 5% of the target standard deviation
EPSILON = b_tr_ours.std().item() * 0.05
print(f"\nTraining perturbed FNO (ε={EPSILON:.4f})...")

# Generate reproducible noise and create perturbed targets
gen = torch.Generator().manual_seed(123)
noise = torch.randn(b_tr_ours.shape, generator=gen) * EPSILON
b_tr_pert = b_tr_ours + noise  # Perturbed targets = original + small noise

# Train a second FNO on the perturbed targets (same architecture, same hyperparams)
model_pert = FNO2d(modes=12, width=32, T_in=T_in, T_out=T_out, depth=4, drop=0.1)
model_pert = train_fno(model_pert, a_tr_ours, b_tr_pert, a_cal, b_cal,
                       epochs=500, ckpt="fno_pert.pt")
model_pert.eval()


Training perturbed FNO (ε=0.0729)...
  ep  0  train=0.6754  val=0.4401
  ep 50  train=0.1955  val=0.2487
  ep100  train=0.1689  val=0.2505
  ep150  train=0.1574  val=0.2512
  ep200  train=0.1484  val=0.2554
  ep250  train=0.1450  val=0.2581
  ep300  train=0.1424  val=0.2612
  ep350  train=0.1403  val=0.2648
  ep400  train=0.1375  val=0.2645
  ep450  train=0.1363  val=0.2675
Best train rel-L2 = 0.1354


FNO2d(
  (fc0): Linear(in_features=12, out_features=32, bias=True)
  (blocks): ModuleList(
    (0-3): 4 x FNOBlock(
      (conv): SpectralConv2d()
      (skip): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc1): Linear(in_features=32, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [5]:
# ============================================================
# CELL 4: TRAIN MA'S FNO + UQNO-E (QUANTILE NETWORK)
# ============================================================
# UQNO must split its budget: 400 for base FNO + 400 for E-network.
# So Ma's base FNO sees FEWER samples → expected to be less accurate.
#
# The E-network predicts the WIDTH of the prediction interval.
# Trained with PINBALL (quantile) LOSS to target the τ-th quantile.
#
# Pinball loss is ASYMMETRIC:
#   - Under-coverage (error > bound): penalized by τ × (error - bound)
#   - Over-coverage  (error < bound): penalized by (1-τ) × (bound - error)
#   With τ=0.99, under-coverage is punished 99x more than over-coverage.
#
# CRITICAL: E-network MUST be trained on data SEPARATE from the base FNO,
# because the base FNO's errors on its own training data would be artificially
# small (overfitting), causing E to learn too-narrow intervals.
# ============================================================

# ----- Train Ma's base FNO (only 400 samples) -----
print(f"\nTraining base FNO (Ma) on {N_TR1_MA} samples...")
model_ma = FNO2d(modes=12, width=32, T_in=T_in, T_out=T_out, depth=4, drop=0.1)
model_ma = train_fno(model_ma, a_tr1_ma, b_tr1_ma, a_cal, b_cal,
                     epochs=500, ckpt="fno_ma_scarce.pt")
model_ma.eval()

# Sanity check: verify prediction accuracy of both FNOs on all splits
print("\nResidual sanity check:")
for tag, mdl, a_, b_ in [
        ("Our FNO  (tr_ours)", model,    a_tr_ours, b_tr_ours),
        ("Our FNO  (cal)    ", model,    a_cal,     b_cal),
        ("Our FNO  (test)   ", model,    a_test,    b_test),
        ("Ma FNO   (tr1_ma) ", model_ma, a_tr1_ma,  b_tr1_ma),
        ("Ma FNO   (cal)    ", model_ma, a_cal,     b_cal),
        ("Ma FNO   (test)   ", model_ma, a_test,    b_test)]:
    with torch.no_grad():
        p_ = mdl(a_.to(DEVICE)).cpu()
    print(f"  rel-L2 {tag}: {rel_l2(p_, b_):.4f}")

# ============================================================
# UQNO-E: Quantile FNO (E-network)
# ============================================================

class QuantileFNO(nn.Module):
    """
    Quantile FNO (E-network) that predicts error bounds.
    Same FNO architecture, but outputs are forced positive (via abs)
    and trained with pinball loss.
    """
    def __init__(self, ref, alpha=0.90):
        super().__init__()
        self.alpha = alpha
        # Create a new FNO with same architecture as the reference model
        self.net   = FNO2d(modes=ref.blocks[0].conv.m1,
                           width=ref.fc0.out_features,
                           T_in =ref.fc0.in_features - 2,
                           T_out=ref.T_out,
                           depth=len(ref.blocks), drop=0.)  # No dropout for E-network

    def forward(self, a):
        """Predict error bound E(a), forced positive with a small floor."""
        raw   = self.net(a).abs()                                          # Ensure positivity
        floor = raw.mean(dim=[1,2,3], keepdim=True).detach() * 0.01 + 1e-6 # Prevent collapse to zero
        return raw + floor

    def qloss(self, e, pred, tgt):
        """
        Pinball (quantile) loss.
          e    = predicted error bound from E-network
          pred = base FNO prediction f(a)
          tgt  = ground truth b
        """
        err = (tgt - pred).abs()       # Actual error |b - f(a)|
        ov  = (err > e).float()        # Under-coverage mask (error exceeds bound)
        un  = 1. - ov                  # Over-coverage mask  (bound exceeds error)
        # Asymmetric: heavily penalize under-coverage (alpha ≈ 0.99)
        return (self.alpha * ov * (err - e) + (1 - self.alpha) * un * (e - err)).mean()


def train_qfno(qfno, base, a_tr, b_tr, epochs=100, bs=10, lr=5e-4):
    """Train the E-network. Base FNO is frozen (no gradient updates)."""
    opt = torch.optim.Adam(qfno.parameters(), lr=lr)
    dl  = DataLoader(TensorDataset(a_tr, b_tr), min(bs, len(a_tr)), shuffle=True)
    qfno.to(DEVICE); base.eval()
    for ep in range(epochs):
        qfno.train(); tot = 0
        for a, b in dl:
            a, b = a.to(DEVICE), b.to(DEVICE)
            with torch.no_grad(): p = base(a)    # Base FNO prediction (frozen)
            e = qfno(a)                           # E-network error bound
            loss = qfno.qloss(e, p, b)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()
        if ep % 20 == 0:
            print(f"  ep{ep:3d}  qloss={tot/len(dl):.5f}")
    return qfno


print(f"\nTraining UQNO-E on {N_TR2_MA} samples (Ma's base FNO)...")
qfno = QuantileFNO(model_ma, alpha=0.99)
qfno = train_qfno(qfno, model_ma, a_tr2_ma, b_tr2_ma, epochs=100)
qfno.eval()


Training base FNO (Ma) on 400 samples...
  ep  0  train=0.8284  val=0.5971
  ep 50  train=0.2003  val=0.2770
  ep100  train=0.1652  val=0.2808
  ep150  train=0.1498  val=0.2819
  ep200  train=0.1381  val=0.2857
  ep250  train=0.1333  val=0.2883
  ep300  train=0.1300  val=0.2913
  ep350  train=0.1268  val=0.2928
  ep400  train=0.1236  val=0.2957
  ep450  train=0.1221  val=0.2970
Best train rel-L2 = 0.1208

Residual sanity check:
  rel-L2 Our FNO  (tr_ours): 0.1383
  rel-L2 Our FNO  (cal)    : 0.2653
  rel-L2 Our FNO  (test)   : 0.2603
  rel-L2 Ma FNO   (tr1_ma) : 0.1267
  rel-L2 Ma FNO   (cal)    : 0.2986
  rel-L2 Ma FNO   (test)   : 0.2922

Training UQNO-E on 400 samples (Ma's base FNO)...
  ep  0  qloss=0.15505
  ep 20  qloss=0.00880
  ep 40  qloss=0.00732
  ep 60  qloss=0.00632
  ep 80  qloss=0.00569


QuantileFNO(
  (net): FNO2d(
    (fc0): Linear(in_features=12, out_features=32, bias=True)
    (blocks): ModuleList(
      (0-3): 4 x FNOBlock(
        (conv): SpectralConv2d()
        (skip): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
      )
    )
    (dropout): Dropout(p=0.0, inplace=False)
    (fc1): Linear(in_features=32, out_features=128, bias=True)
    (fc2): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [6]:
# ============================================================
# CELL 5: LAPLACE PRECISION + PERTURBATION SIGMA + CONFORMAL UTILITIES
# ============================================================
# This cell computes all the uncertainty scales needed for each method
# and defines the conformal prediction utility functions.
# ============================================================

# ============================================================
# 5a. LAPLACE PRECISION
# ============================================================
# Diagonal Laplace approximation estimates posterior uncertainty as:
#   Var(a*) = Σ_d  φ_d(a*)²  /  precision_d
#
# where φ(a) is the feature vector (from forward_with_features),
# and precision_d = 1 + Σ_i φ_d(a_i)²  over all training samples.
#
# High precision (many similar training features seen) → low uncertainty.
# Low precision (novel features at test time) → high uncertainty.
# ============================================================

feat_dim = len(model.blocks) * model.fc0.out_features  # 4 blocks × 32 width = 128
print(f"\nFitting Laplace precision (feat_dim={feat_dim})...")

# Accumulate precision = 1 + Σ_i φ_d(a_i)²  over training set
precision = torch.ones(feat_dim)  # Start with 1 (prior)
with torch.no_grad():
    for i in range(0, len(a_tr_ours), 10):
        _, phi = model.forward_with_features(a_tr_ours[i:i+10].to(DEVICE))
        precision += (phi ** 2).sum(dim=0).cpu()
print(f"  precision (ours): min={precision.min():.1f}  max={precision.max():.1f}")

# Same for Ma's FNO (trained on fewer samples → lower precision → higher uncertainty)
precision_ma = torch.ones(feat_dim)
with torch.no_grad():
    for i in range(0, len(a_tr1_ma), 10):
        _, phi = model_ma.forward_with_features(a_tr1_ma[i:i+10].to(DEVICE))
        precision_ma += (phi ** 2).sum(dim=0).cpu()
print(f"  precision (Ma)  : min={precision_ma.min():.1f}  max={precision_ma.max():.1f}")

# ============================================================
# 5b. SPATIAL SMOOTHING FUNCTION
# ============================================================
# Raw uncertainty maps can be noisy pixel-to-pixel.
# This applies a box filter (average pooling with reflection padding)
# to produce smoother, more stable uncertainty estimates.
# ============================================================

def smooth_J(J_raw, kernel_size=5):
    """
    Spatial smoothing via average pooling.
      J_raw: (N, H, W, T) — raw uncertainty map
      Returns: (N, H, W, T) — smoothed map
    """
    N, H, W, T = J_raw.shape
    # Reshape: treat each time step as a 1-channel image
    J = J_raw.permute(0, 3, 1, 2).reshape(N*T, 1, H, W)
    pad = kernel_size // 2
    J_smooth = F.avg_pool2d(
        F.pad(J, [pad]*4, mode='reflect'),  # Reflect-pad edges to avoid boundary artifacts
        kernel_size, stride=1                # Average over kernel×kernel neighborhood
    )
    return J_smooth.reshape(N, T, H, W).permute(0, 2, 3, 1)

# Pre-compute base FNO predictions and errors on calibration set
model.eval()
pred_cal_jac = []
with torch.no_grad():
    for i in range(0, len(a_cal), 10):
        pred_cal_jac.append(model(a_cal[i:i+10].to(DEVICE)).cpu())
pred_cal_jac   = torch.cat(pred_cal_jac)          # (N_cal, 64, 64, T_out)
err_cal_tensor = (b_cal - pred_cal_jac).abs()      # Pointwise absolute error

# ============================================================
# 5c. PERTURBATION SIGMA
# ============================================================
# σ_pert(a) = |f(a) − f_pert(a)|  (smoothed + floor)
#
# Compute for calibration and test sets:
#   1. Pointwise disagreement: |f(a) − f_pert(a)|
#   2. Spatial smoothing: 15×15 average pooling
#   3. Floor stabilization: clamp to ≥ τ₀ = 0.1 × median(disagreement on train)
# ============================================================

print("\nComputing perturbation σ...")
SMOOTH_KS_PERT = 15  # 15×15 smoothing window (Eq. 4.4 in paper)

# Get predictions from both FNOs on calibration and test sets
pred_cal_orig, pred_cal_pert = [], []
pred_test_orig, pred_test_pert = [], []
model.eval(); model_pert.eval()
with torch.no_grad():
    for i in range(0, len(a_cal), 10):
        batch = a_cal[i:i+10].to(DEVICE)
        pred_cal_orig.append(model(batch).cpu())
        pred_cal_pert.append(model_pert(batch).cpu())
    for i in range(0, len(a_test), 10):
        batch = a_test[i:i+10].to(DEVICE)
        pred_test_orig.append(model(batch).cpu())
        pred_test_pert.append(model_pert(batch).cpu())

pred_cal_orig = torch.cat(pred_cal_orig)
pred_cal_pert = torch.cat(pred_cal_pert)
pred_test_orig = torch.cat(pred_test_orig)
pred_test_pert = torch.cat(pred_test_pert)

# Pointwise difference → smooth → sigma (Eq. 4.3–4.5)
sigma_cal_pert  = smooth_J((pred_cal_orig - pred_cal_pert).abs(), kernel_size=SMOOTH_KS_PERT)
sigma_test_pert = smooth_J((pred_test_orig - pred_test_pert).abs(), kernel_size=SMOOTH_KS_PERT)

# Compute floor: 10% of median sigma on training set (Eq. 4.5)
# This prevents the denominator in the conformal score from becoming arbitrarily small
sigma_tr_pert = []
with torch.no_grad():
    for i in range(0, len(a_tr_ours), 10):
        batch = a_tr_ours[i:i+10].to(DEVICE)
        sigma_tr_pert.append((model(batch) - model_pert(batch)).abs().cpu())
sigma_tr_pert = torch.cat(sigma_tr_pert)
PERT_FLOOR = sigma_tr_pert.median().item() * 0.1

# Clamp sigma to be at least PERT_FLOOR
sigma_cal_pert  = sigma_cal_pert.clamp(min=PERT_FLOOR)
sigma_test_pert = sigma_test_pert.clamp(min=PERT_FLOOR)

print(f"  σ_pert: mean={sigma_cal_pert.mean():.6f}  FLOOR={PERT_FLOOR:.6f}  "
      f"clamp%={(sigma_cal_pert == PERT_FLOOR).float().mean()*100:.1f}%")

# ============================================================
# 5d. CONFORMAL PREDICTION UTILITIES
# ============================================================
# Split conformal prediction gives distribution-free prediction intervals
# with guaranteed coverage.
#
# Steps:
#   1. score_i = max_k( |error_k| / uncertainty_k ) for each calibration sample
#      → single scalar per sample (max-type score for simultaneous coverage)
#   2. λ = ceil((n+1)(1-α))/n quantile of scores → conformal threshold
#   3. Prediction interval = pred ± λ × uncertainty
#   4. Coverage guarantee: P(true ∈ interval simultaneously ∀k) ≥ 1 − α
# ============================================================

def nc_score(true_err, E_est):
    """
    Nonconformity score = max ratio of true error to estimated uncertainty.
    A high score = uncertainty was too small somewhere (under-coverage).

    S(a, u) = max_k |u^k − G_hat(a)^k| / σ(a)^k     (Eq. 4.6 in paper)
    """
    ratios = (true_err / E_est.clamp(min=1e-8)).flatten().numpy()
    return float(np.max(ratios))


def conformal_lambda(scores, n, alpha):
    """
    Conformal threshold λ = quantile of scores at level ⌈(n+1)(1−α)⌉/n.
    The ceiling provides the finite-sample coverage guarantee.

    Q_hat_{1-α} = S_{(⌈(n+1)(1-α)⌉)}     (Eq. 4.7 in paper)
    """
    level = min(math.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(np.array(scores), level))


def coverage_stats(pred, tgt, radius):
    """
    Compute coverage statistics for prediction intervals.

    Returns:
        cal_pct  : fraction of test samples with SIMULTANEOUS coverage (all pixels covered)
        bw       : mean bandwidth (average interval width)
        per_fn   : per-sample pixel-level coverage fraction
    """
    cov     = ((tgt - pred).abs() <= radius).float()
    per_fn  = cov.mean(dim=[1, 2, 3])                        # Per-sample pixel coverage
    cal_pct = (per_fn == 1).float().mean().item()             # Fraction with 100% coverage
    bw      = radius.mean().item()                            # Mean interval width
    return cal_pct, bw, per_fn.numpy()


def batch_predict(mdl, a_data, bs=10):
    """Run model in batches, return predictions and features."""
    ps, fs = [], []
    mdl.eval()
    for i in range(0, len(a_data), bs):
        with torch.no_grad():
            p, f = mdl.forward_with_features(a_data[i:i+bs].to(DEVICE))
        ps.append(p.cpu()); fs.append(f.cpu())
    return torch.cat(ps), torch.cat(fs)


Fitting Laplace precision (feat_dim=128)...
  precision (ours): min=9.4  max=9108.1
  precision (Ma)  : min=1.6  max=4296.1

Computing perturbation σ...
  σ_pert: mean=0.194909  FLOOR=0.007192  clamp%=0.0%


In [7]:
# ============================================================
# CELL 6: UQ METHOD RUNNERS
# ============================================================
# Each run_* function:
#   1. Computes the method-specific uncertainty σ on calibration data
#   2. Computes nonconformity scores on calibration data
#   3. Finds the conformal threshold λ at the (1−α) quantile
#   4. Evaluates on test data: coverage and bandwidth
# ============================================================

# ============================================================
# 6a. MC DROPOUT
# ============================================================
# Keeps dropout ON at test time; runs n_mc=20 stochastic forward passes.
# Mean and variance are computed via Welford's numerically-stable algorithm.
# High variance across passes = high uncertainty.
#
# Reference: Gal & Ghahramani, "Dropout as a Bayesian Approximation" (ICML 2016)
# ============================================================

def run_mcd(alpha, n_mc=20, bs=10):
    """MC Dropout UQ with conformal calibration."""
    def get_mc_stats(a_data):
        N = len(a_data)
        mean = torch.zeros(N, S, S, T_out)  # Welford running mean
        M2   = torch.zeros(N, S, S, T_out)  # Welford sum-of-squared-deviations
        for k in range(1, n_mc + 1):
            ps = []
            for i in range(0, N, bs):
                with torch.no_grad():
                    model.train()  # CRITICAL: keeps dropout active during inference
                    ps.append(model(a_data[i:i+bs].to(DEVICE)).cpu())
            sample  = torch.cat(ps)
            delta_k = sample - mean                  # Deviation from current mean
            mean    = mean + delta_k / k             # Update mean incrementally
            M2      = M2 + delta_k * (sample - mean) # Welford M2 accumulator
        model.eval()
        std = torch.sqrt((M2 / (n_mc - 1)).clamp(min=0) + 1e-12)  # Sample std dev
        return mean, std

    n = len(a_cal)
    pred_c, std_c = get_mc_stats(a_cal)
    te_c = (b_cal - pred_c).abs()
    sc   = [nc_score(te_c[i], std_c[i]) for i in range(n)]
    lam  = conformal_lambda(sc, n, alpha)

    pred_t, std_t = get_mc_stats(a_test)
    radius = lam * std_t
    cal_pct, bw, per_fn = coverage_stats(pred_t, b_test, radius)
    return dict(cal_pct=cal_pct, bw=bw, per_fn=per_fn,
                pred=pred_t, radius=radius, lam=lam)


# ============================================================
# 6b. LAPLACE
# ============================================================
# σ²(a*) = Σ_d φ_d(a*)² / precision_d
# Produces a SCALAR uncertainty per sample (broadcast to full 64×64×T_out grid).
# ============================================================

def run_laplace(alpha, bs=10):
    """Laplace approximation UQ with conformal calibration."""
    def get_lap_std(a_data):
        stds = []
        for i in range(0, len(a_data), bs):
            with torch.no_grad():
                _, phi = model.forward_with_features(a_data[i:i+bs].to(DEVICE))
                var = (phi.cpu() ** 2 / precision).sum(dim=-1)  # (batch,) scalar variance
                std = torch.sqrt(var + 1e-8)
                stds.append(std.view(-1, 1, 1, 1).expand(-1, S, S, T_out).clone())
        return torch.cat(stds)

    n = len(a_cal)
    pred_c, _ = batch_predict(model, a_cal, bs)
    std_c     = get_lap_std(a_cal)
    te_c      = (b_cal - pred_c).abs()
    sc        = [nc_score(te_c[i], std_c[i]) for i in range(n)]
    lam       = conformal_lambda(sc, n, alpha)

    pred_t, _ = batch_predict(model, a_test, bs)
    std_t     = get_lap_std(a_test)
    radius    = lam * std_t
    cal_pct, bw, per_fn = coverage_stats(pred_t, b_test, radius)
    return dict(cal_pct=cal_pct, bw=bw, per_fn=per_fn,
                pred=pred_t, radius=radius, lam=lam)


# ============================================================
# 6c. UQNO (Ma et al.)
# ============================================================
# Uses model_ma (400 samples) + qfno (E-network, 400 samples).
# ============================================================

def run_uqno(alpha, bs=10):
    """UQNO (Ma et al.) with conformal calibration."""
    n = len(a_cal)
    pred_c, e_c = [], []
    for i in range(0, n, bs):
        a = a_cal[i:i+bs].to(DEVICE)
        with torch.no_grad():
            pred_c.append(model_ma(a).cpu())
            e_c.append(qfno(a).cpu())
    pred_c = torch.cat(pred_c); e_c = torch.cat(e_c)
    te  = (b_cal - pred_c).abs()
    sc  = [nc_score(te[i], e_c[i]) for i in range(n)]
    lam = conformal_lambda(sc, n, alpha)

    pred_t, e_t = [], []
    for i in range(0, len(a_test), bs):
        a = a_test[i:i+bs].to(DEVICE)
        with torch.no_grad():
            pred_t.append(model_ma(a).cpu())
            e_t.append(qfno(a).cpu())
    pred_t = torch.cat(pred_t); e_t = torch.cat(e_t)
    radius = lam * e_t
    cal_pct, bw, per_fn = coverage_stats(pred_t, b_test, radius)
    return dict(cal_pct=cal_pct, bw=bw, per_fn=per_fn,
                pred=pred_t, radius=radius, lam=lam)


# ============================================================
# 6d. PERTURBATION (OURS)
# ============================================================
# Uses pre-computed sigma_pert from Cell 5c.
# This is the simplest runner — σ is already available.
# ============================================================

def run_perturbation(alpha):
    """Perturbation-based UQ with conformal calibration."""
    n  = len(a_cal)
    sc = [nc_score(err_cal_tensor[i], sigma_cal_pert[i]) for i in range(n)]
    lam = conformal_lambda(sc, n, alpha)

    pred_t, _ = batch_predict(model, a_test, 10)
    radius    = lam * sigma_test_pert
    cal_pct, bw, per_fn = coverage_stats(pred_t, b_test, radius)
    return dict(cal_pct=cal_pct, bw=bw, per_fn=per_fn,
                pred=pred_t, radius=radius, lam=lam)


# ============================================================
# 6e. UNSCALED BASELINE (σ ≡ 1, constant-width bands)
# ============================================================
# This ablation baseline isolates the value of the perturbation-derived
# local scale by setting σ(a)^k = 1 for all k. The resulting conformal
# bands are constant-width everywhere.
# ============================================================

def run_nosigma(alpha, bs=10):
    """Unscaled conformal: σ=1, score = max|r|, radius = λ (constant)."""
    n = len(a_cal)
    pred_c, _ = batch_predict(model, a_cal, bs)
    te_c = (b_cal - pred_c).abs()
    sc = [te_c[i].max().item() for i in range(n)]  # max|r| since σ=1
    lam = conformal_lambda(sc, n, alpha)

    pred_t, _ = batch_predict(model, a_test, bs)
    radius = torch.full_like(pred_t, lam)           # constant radius everywhere
    cal_pct, bw, per_fn = coverage_stats(pred_t, b_test, radius)
    return dict(cal_pct=cal_pct, bw=bw, per_fn=per_fn,
                pred=pred_t, radius=radius, lam=lam)

In [8]:
# ============================================================
# CELL 7: RUN ALL METHODS (SINGLE SPLIT) + SUMMARY TABLE
# ============================================================
# Evaluate at 5 significance levels: α = 0.02, 0.04, 0.06, 0.08, 0.10
# Lower α = stricter coverage target = wider intervals needed
# ============================================================

alpha_vals = [0.02, 0.04, 0.06, 0.08, 0.10]
ALPHA_HIGH = 0.04  # 96% coverage target
ALPHA_LOW  = 0.10  # 90% coverage target

RES = {m: {} for m in METHODS}
for a_ in alpha_vals:
    print(f"\n--- alpha={a_:.2f} ---")
    RES["MCDropout"][a_] = run_mcd(a_)
    RES["Laplace"][a_]   = run_laplace(a_)
    RES["UQNO"][a_]      = run_uqno(a_)
    RES["NoSigma"][a_]   = run_nosigma(a_)
    RES["Perturbation"][a_] = run_perturbation(a_)
    for m in METHODS:
        r  = RES[m][a_]
        ok = "OK" if r['cal_pct'] >= 1-a_ else "FAIL"
        print(f"  {LABELS[m]:<28}  cal%={r['cal_pct']*100:.2f}%"
              f"  bw={r['bw']:.5f}  lam={r['lam']:.3f}  [{ok}]")

# ============================================================
# Summary Table (single split — for quick verification only)
# ============================================================
sep = "-" * 96
print("\n" + sep)
print(f"  DATA-SCARCE COMPARISON — 2D NS UQ  (target = 1-alpha per level)")
print(f"  Our FNO: {N_TR_OURS} samples  |  Ma FNO: {N_TR1_MA}+{N_TR2_MA} samples  |"
      f"  cal={N_CAL}  test={N_TEST}")
print(sep)

print(f"\n{'Method':<28}  {'alpha':>6}  {'Cal%':>7}  {'BW':>10}  {'Lam':>8}  Pass")
print("-" * 74)
for m in METHODS:
    for a_ in alpha_vals:
        r  = RES[m][a_]
        ok = "  OK" if r['cal_pct'] >= 1-a_ else "  --"
        print(f"{LABELS[m]:<28}  {a_:>6.2f}  "
              f"{r['cal_pct']*100:>6.1f}%  {r['bw']:>10.5f}"
              f"  {r['lam']:>8.3f}{ok}")
    print("-" * 74)


--- alpha=0.02 ---
  MCDropout                     cal%=100.00%  bw=11.30710  lam=128.634  [OK]
  Laplace                       cal%=92.50%  bw=4.32698  lam=10.877  [FAIL]
  UQNO                          cal%=100.00%  bw=37.16220  lam=47.535  [OK]
  Perturbation(ours)            cal%=98.50%  bw=4.22162  lam=22.209  [OK]
  Unscaled ($\sigma\equiv 1$)   cal%=98.50%  bw=4.37602  lam=4.376  [OK]

--- alpha=0.04 ---
  MCDropout                     cal%=99.00%  bw=7.91760  lam=90.094  [OK]
  Laplace                       cal%=89.00%  bw=4.13369  lam=10.391  [FAIL]
  UQNO                          cal%=98.00%  bw=19.26243  lam=24.639  [OK]
  Perturbation(ours)            cal%=97.00%  bw=3.75659  lam=19.763  [OK]
  Unscaled ($\sigma\equiv 1$)   cal%=98.50%  bw=4.26609  lam=4.266  [OK]

--- alpha=0.06 ---
  MCDropout                     cal%=93.00%  bw=7.48855  lam=85.191  [FAIL]
  Laplace                       cal%=89.00%  bw=4.11076  lam=10.334  [FAIL]
  UQNO                          cal%=92.

In [9]:
# ============================================================
# CELL 8: 1000-RESHUFFLE EXPERIMENT WITH ± STANDARD ERRORS
# ============================================================
#
# PURPOSE:
# The previous cell evaluated on a SINGLE cal/test split, which could
# be "lucky" or "unlucky". Here we fix the trained models and repeatedly
# RESHUFFLE the remaining data into new cal/test sets (1000 times)
# to verify that conformal coverage holds ON AVERAGE.
#
# REVIEWER REQUEST ADDRESSED:
# "Right now the table reports mean coverage and bandwidth, but no
#  standard errors, no variance over reshuffles."
# → This cell reports mean ± SE for both coverage and bandwidth.
#
# MEMORY-EFFICIENT TRICK:
# Instead of storing full (64,64,10) tensors per sample, we pre-compute
# two SCALARS per sample:
#   score_i     = max(|error_i| / σ_i)   — nonconformity score
#   mean_sigma_i = mean(σ_i)              — average uncertainty
# Then each reshuffle only shuffles small arrays — no GPU needed.
# ============================================================

np.random.seed(42)
N_REPEATS = 1000   # Number of random reshuffles
N_CAL     = 200    # Calibration set size per reshuffle
N_TEST    = 200    # Test set size per reshuffle
alpha_vals = [0.02, 0.04, 0.06, 0.08, 0.10]

# Pool = all data NOT used for our base FNO training
# These 400 samples will be reshuffled into cal/test splits
pool_a = a_all[N_TR_OURS:]
pool_b = b_all[N_TR_OURS:]
N_pool = len(pool_a)   # 1200 − 800 = 400
print(f"Pool size: {N_pool}  (cal={N_CAL} + test={N_TEST})")

# ============================================================
# Pre-compute per-sample scores and mean_sigmas for entire pool
# ============================================================

def precompute_scores(pool_a, pool_b, method, bs=5):
    """
    For each pool sample, compute:
      score      = max(|error| / σ)   → scalar  (nonconformity score)
      mean_sigma = mean(σ)            → scalar  (for bandwidth computation)

    Processes in mini-batches for GPU memory efficiency.
    Returns two numpy arrays of shape (N_pool,).
    """
    scores = []
    mean_sigmas = []

    for i in range(0, len(pool_a), bs):
        a_batch = pool_a[i:i+bs].to(DEVICE)
        b_batch = pool_b[i:i+bs]

        with torch.no_grad():
            if method == "Laplace":
                # Laplace: σ from feature-space precision (scalar per sample)
                pred, phi = model.forward_with_features(a_batch)
                pred = pred.cpu()
                var = (phi.cpu() ** 2 / precision).sum(dim=-1)
                std = torch.sqrt(var + 1e-8)
                sigma = std.view(-1, 1, 1, 1).expand(-1, S, S, T_out).clone()

            elif method == "MCDropout":
                # MC Dropout: Welford online mean/var over 20 stochastic passes
                n_mc = 20
                B = len(a_batch)
                mean_mc = torch.zeros(B, S, S, T_out)
                M2_mc   = torch.zeros(B, S, S, T_out)
                for k in range(1, n_mc + 1):
                    model.train()   # Activate dropout
                    with torch.no_grad():
                        sample = model(a_batch).cpu()
                    delta_k = sample - mean_mc
                    mean_mc = mean_mc + delta_k / k
                    M2_mc   = M2_mc + delta_k * (sample - mean_mc)
                model.eval()
                pred = mean_mc
                sigma = torch.sqrt((M2_mc / (n_mc - 1)).clamp(min=0) + 1e-12)

            elif method == "UQNO":
                # UQNO: uses Ma's base FNO + E-network
                pred = model_ma(a_batch).cpu()
                sigma = qfno(a_batch).cpu()

            elif method == "NoSigma":
                # Unscaled: σ ≡ 1
                pred = model(a_batch).cpu()
                sigma = torch.ones_like(pred)

            elif method == "Perturbation":
                # Perturbation: |f(a) − f_pert(a)| smoothed + floor
                pred = model(a_batch).cpu()
                sigma = smooth_J((model(a_batch) - model_pert(a_batch)).abs().cpu(),
                                 kernel_size=SMOOTH_KS_PERT).clamp(min=PERT_FLOOR)

        # Compute score = max ratio, and mean_sigma per sample
        err = (b_batch - pred).abs()
        ratios = err / sigma.clamp(min=1e-8)
        batch_scores = ratios.reshape(len(a_batch), -1).max(dim=1).values
        batch_mean_sig = sigma.mean(dim=[1, 2, 3])

        scores.append(batch_scores)
        mean_sigmas.append(batch_mean_sig)

    return torch.cat(scores).numpy(), torch.cat(mean_sigmas).numpy()


# Pre-compute for all 5 methods
for m_name in METHODS:
    print(f"Pre-computing scores for {m_name}...")

all_scores = {}
all_msigma = {}
print("Pre-computing scores for Laplace...")
all_scores["Laplace"], all_msigma["Laplace"] = precompute_scores(pool_a, pool_b, "Laplace", bs=20)
print("Pre-computing scores for MCDropout...")
all_scores["MCDropout"], all_msigma["MCDropout"] = precompute_scores(pool_a, pool_b, "MCDropout", bs=20)
print("Pre-computing scores for UQNO...")
all_scores["UQNO"], all_msigma["UQNO"] = precompute_scores(pool_a, pool_b, "UQNO", bs=20)
print("Pre-computing scores for NoSigma...")
all_scores["NoSigma"], all_msigma["NoSigma"] = precompute_scores(pool_a, pool_b, "NoSigma", bs=20)
print("Pre-computing scores for Perturbation...")
all_scores["Perturbation"], all_msigma["Perturbation"] = precompute_scores(pool_a, pool_b, "Perturbation", bs=20)

print(f"\nMemory: {len(METHODS) * N_pool * 4 * 2 / 1024:.1f} KB total "
      f"({len(METHODS)} methods × {N_pool} × 2 floats)")
print("Starting 1000 repeats...\n")


# ============================================================
# Main reshuffle loop
# ============================================================
# Each repeat: random cal/test split → compute λ → check coverage + bandwidth

results = {m: {a: {"cov": [], "bw": []} for a in alpha_vals}
           for m in METHODS}

for rep in range(N_REPEATS):
    perm = np.random.permutation(N_pool)
    idx_cal  = perm[:N_CAL]
    idx_test = perm[N_CAL:N_CAL + N_TEST]

    for alpha in alpha_vals:
        # Conformal quantile level with finite-sample correction
        level = min(math.ceil((N_CAL + 1) * (1 - alpha)) / N_CAL, 1.0)

        for m in METHODS:
            sc_cal  = all_scores[m][idx_cal]
            sc_test = all_scores[m][idx_test]
            ms_test = all_msigma[m][idx_test]

            lam = float(np.quantile(sc_cal, level))

            # Coverage: fraction of test samples with score ≤ λ
            #   score ≤ λ ⟺ all output coordinates are covered simultaneously
            cov = np.mean(sc_test <= lam)
            # Bandwidth: λ × mean(σ) = average half-width of prediction band
            bw  = lam * np.mean(ms_test)

            results[m][alpha]["cov"].append(cov)
            results[m][alpha]["bw"].append(bw)

    if (rep + 1) % 200 == 0:
        print(f"  {rep+1}/{N_REPEATS} done")

# ============================================================
# Summary table: mean ± SE over 1000 reshuffles
# ============================================================
# SE = standard error of the mean = std / √N_REPEATS
# This is what goes into the paper's Table 1.
# ============================================================

print("\n" + "=" * 95)
print(f"  TABLE 1 (PAPER): AVERAGE ± SE OVER {N_REPEATS} RESHUFFLES")
print(f"  N_tr={N_TR_OURS}  N_cal={N_CAL}  N_test={N_TEST}  pool={N_pool}")
print("=" * 95)
print(f"{'Method':<16} {'alpha':>6} | {'Avg Cov%':>10} {'±SE':>8} | "
      f"{'Avg BW':>10} {'±SE':>10} | {'±Std(BW)':>10}")
print("-" * 95)

for m in METHODS:
    for a_ in alpha_vals:
        covs = np.array(results[m][a_]["cov"])
        bws  = np.array(results[m][a_]["bw"])
        n_rep = len(covs)
        cov_se = np.std(covs) * 100 / np.sqrt(n_rep)   # SE of coverage (%)
        bw_se  = np.std(bws) / np.sqrt(n_rep)           # SE of bandwidth
        bw_std = np.std(bws)                              # Raw std of bandwidth
        print(f"{LABELS[m]:<16} {a_:>6.2f} | "
              f"{np.mean(covs)*100:>8.2f}% ±{cov_se:>5.3f}% | "
              f"{np.mean(bws):>10.5f} ±{bw_se:>8.5f} | "
              f"±{bw_std:>8.5f}")
    print("-" * 95)

Pool size: 400  (cal=200 + test=200)
Pre-computing scores for MCDropout...
Pre-computing scores for Laplace...
Pre-computing scores for UQNO...
Pre-computing scores for Perturbation...
Pre-computing scores for NoSigma...
Pre-computing scores for Laplace...
Pre-computing scores for MCDropout...
Pre-computing scores for UQNO...
Pre-computing scores for NoSigma...
Pre-computing scores for Perturbation...

Memory: 15.6 KB total (5 methods × 400 × 2 floats)
Starting 1000 repeats...

  200/1000 done
  400/1000 done
  600/1000 done
  800/1000 done
  1000/1000 done

  TABLE 1 (PAPER): AVERAGE ± SE OVER 1000 RESHUFFLES
  N_tr=800  N_cal=200  N_test=200  pool=400
Method            alpha |   Avg Cov%      ±SE |     Avg BW        ±SE |   ±Std(BW)
-----------------------------------------------------------------------------------------------
MCDropout          0.02 |    97.96% ±0.044% |   10.28853 ± 0.02878 | ± 0.91006
MCDropout          0.04 |    95.95% ±0.061% |    8.53013 ± 0.01509 | ± 0.47708
M

In [10]:
# ============================================================
# CELL 9: ABLATION STUDY — SMOOTHING AND FLOOR
# ============================================================
#
# REVIEWER REQUEST ADDRESSED:
# "They will also want to know whether the gains survive [...] removal
#  of smoothing/floor stabilization."
#
# We test 4 configurations, each requiring only re-computation of σ
# on the pool (no model retraining needed):
#
#   (1) Full method:  K=15, floor = τ₀     ← paper default
#   (2) No smoothing: K=1,  floor = τ₀     ← raw disagreement + floor
#   (3) No floor:     K=15, floor = 0       ← smoothed but no floor
#   (4) Neither:      K=1,  floor = 0       ← raw disagreement only
#
# For each configuration, we re-compute perturbation scores on the
# entire pool and run 1000 reshuffles to get mean ± SE.
#
# KEY INSIGHT: This experiment does NOT require retraining any model.
# We reuse the trained model and model_pert, and only change the
# post-processing (smoothing kernel size and floor value).
# ============================================================

print("=" * 90)
print("  ABLATION STUDY: Effect of Spatial Smoothing and Floor Stabilization")
print("=" * 90)

def precompute_perturbation_ablation(pool_a, pool_b, smooth_ks, floor_val, bs=20):
    """
    Compute per-sample (score, mean_sigma) for the perturbation method
    with specified smoothing kernel size and floor value.

    This allows testing different ablation configurations without
    retraining the models.

    Parameters:
        pool_a    : pool input data
        pool_b    : pool target data
        smooth_ks : spatial smoothing kernel size (K=1 means no smoothing)
        floor_val : minimum uncertainty floor (0 means no floor)
        bs        : mini-batch size

    Returns:
        scores     : (N_pool,) nonconformity scores
        mean_sigmas: (N_pool,) mean sigma per sample
    """
    scores = []
    mean_sigmas = []

    for i in range(0, len(pool_a), bs):
        a_batch = pool_a[i:i+bs].to(DEVICE)
        b_batch = pool_b[i:i+bs]

        with torch.no_grad():
            pred = model(a_batch).cpu()
            pred_p = model_pert(a_batch).cpu()

        # Raw disagreement: |f(a) − f_pert(a)|
        raw_diff = (pred - pred_p).abs()

        # Smoothing (K=1 means no smoothing since avg_pool with kernel=1 is identity)
        if smooth_ks > 1:
            sigma = smooth_J(raw_diff, kernel_size=smooth_ks)
        else:
            sigma = raw_diff

        # Floor (0 means no floor; use tiny epsilon to avoid division by zero)
        sigma = sigma.clamp(min=max(floor_val, 1e-12))

        # Scores
        err = (b_batch - pred).abs()
        ratios = err / sigma
        batch_scores = ratios.reshape(len(a_batch), -1).max(dim=1).values
        batch_mean_sig = sigma.mean(dim=[1, 2, 3])

        scores.append(batch_scores)
        mean_sigmas.append(batch_mean_sig)

    return torch.cat(scores).numpy(), torch.cat(mean_sigmas).numpy()


def run_reshuffle(scores, msigma, N_pool, N_CAL, N_TEST, alpha_vals, N_REPEATS=1000):
    """
    Run N_REPEATS random cal/test reshuffles and collect coverage + bandwidth.

    This is a generic helper used by both the ablation study and the
    noise sensitivity experiment.

    Returns:
        results_cov : {alpha: [cov_1, ..., cov_N_REPEATS]}
        results_bw  : {alpha: [bw_1, ..., bw_N_REPEATS]}
    """
    results_cov = {a: [] for a in alpha_vals}
    results_bw  = {a: [] for a in alpha_vals}

    for rep in range(N_REPEATS):
        perm = np.random.permutation(N_pool)
        idx_cal  = perm[:N_CAL]
        idx_test = perm[N_CAL:N_CAL + N_TEST]

        for alpha in alpha_vals:
            level = min(math.ceil((N_CAL + 1) * (1 - alpha)) / N_CAL, 1.0)
            lam = float(np.quantile(scores[idx_cal], level))
            cov = np.mean(scores[idx_test] <= lam)
            bw  = lam * np.mean(msigma[idx_test])

            results_cov[alpha].append(cov)
            results_bw[alpha].append(bw)

    return results_cov, results_bw


# ============================================================
# Define ablation configurations
# ============================================================
ABLATION_CONFIGS = {
    "Full (K=15, floor)":        {"smooth_ks": 15, "floor_val": PERT_FLOOR},
    "No smoothing (K=1, floor)": {"smooth_ks": 1,  "floor_val": PERT_FLOOR},
    "No floor (K=15, floor=0)":  {"smooth_ks": 15, "floor_val": 0.0},
    "Neither (K=1, floor=0)":    {"smooth_ks": 1,  "floor_val": 0.0},
}

# Run ablation for each configuration
ablation_results = {}
for name, cfg in ABLATION_CONFIGS.items():
    print(f"\n  Config: {name}  (K={cfg['smooth_ks']}, floor={cfg['floor_val']:.6f})")
    sc, ms = precompute_perturbation_ablation(
        pool_a, pool_b,
        smooth_ks=cfg["smooth_ks"],
        floor_val=cfg["floor_val"],
        bs=20
    )
    cov_results, bw_results = run_reshuffle(
        sc, ms, N_pool, N_CAL, N_TEST, alpha_vals, N_REPEATS=1000
    )
    ablation_results[name] = {"cov": cov_results, "bw": bw_results}
    print(f"    Done.")

# ============================================================
# Print Ablation Table (Table 2 in paper)
# ============================================================
print("\n" + "=" * 105)
print(f"  TABLE 2 (PAPER): ABLATION — SMOOTHING AND FLOOR ({N_REPEATS} reshuffles)")
print("=" * 105)
for a_ in [0.02, 0.04, 0.06]:
    print(f"\n  α = {a_} (target coverage: {(1-a_)*100:.0f}%)")
    print(f"  {'Configuration':<30} {'Cov%':>8} {'±SE':>8}   {'BW':>10} {'±SE':>10}")
    print(f"  {'-'*72}")
    for name in ABLATION_CONFIGS:
        covs = np.array(ablation_results[name]["cov"][a_])
        bws  = np.array(ablation_results[name]["bw"][a_])
        n = len(covs)
        print(f"  {name:<30} {covs.mean()*100:>7.2f}% ±{covs.std()*100/np.sqrt(n):>5.3f}%"
              f"   {bws.mean():>9.4f} ±{bws.std()/np.sqrt(n):>8.5f}")

  ABLATION STUDY: Effect of Spatial Smoothing and Floor Stabilization

  Config: Full (K=15, floor)  (K=15, floor=0.007192)
    Done.

  Config: No smoothing (K=1, floor)  (K=1, floor=0.007192)
    Done.

  Config: No floor (K=15, floor=0)  (K=15, floor=0.000000)
    Done.

  Config: Neither (K=1, floor=0)  (K=1, floor=0.000000)
    Done.

  TABLE 2 (PAPER): ABLATION — SMOOTHING AND FLOOR (1000 reshuffles)

  α = 0.02 (target coverage: 98%)
  Configuration                      Cov%      ±SE           BW        ±SE
  ------------------------------------------------------------------------
  Full (K=15, floor)               98.07% ±0.042%      4.1806 ± 0.00563
  No smoothing (K=1, floor)        98.09% ±0.043%     87.5728 ± 0.05783
  No floor (K=15, floor=0)         98.02% ±0.045%      4.1734 ± 0.00579
  Neither (K=1, floor=0)           98.10% ±0.046%   21328916.0000 ±1803147.67164

  α = 0.04 (target coverage: 96%)
  Configuration                      Cov%      ±SE           BW        ±S

In [11]:
# ============================================================
# CELL 10: NOISE SENSITIVITY — VARYING σ_ε
# ============================================================
#
# REVIEWER REQUEST ADDRESSED:
# "They will also want to know whether the gains survive different
#  perturbation noise levels."
#
# We sweep σ_ε ∈ {0.01, 0.02, 0.05, 0.10, 0.20} × std(u_train).
# The default in the paper is 0.05.
#
# Each noise level REQUIRES RETRAINING the perturbed FNO (model_pert),
# since the perturbed labels change. The base FNO (model) stays the same.
#
# For each noise level:
#   1. Generate perturbed labels with the new σ_ε
#   2. Train a new model_pert on these labels (500 epochs)
#   3. Compute floor τ₀ on training set
#   4. Pre-compute scores on pool
#   5. Run 1000 reshuffles
#
# Expected runtime: ~5 noise levels × ~5 min/training = ~25 min total
# ============================================================

NOISE_MULTIPLIERS = [0.01, 0.02, 0.05, 0.10, 0.20]

print("\n" + "=" * 90)
print("  NOISE SENSITIVITY: Varying Perturbation Magnitude σ_ε")
print(f"  σ_ε ∈ {NOISE_MULTIPLIERS} × std(u_train)")
print("=" * 90)

noise_results = {}

for mult in NOISE_MULTIPLIERS:
    eps = b_tr_ours.std().item() * mult
    print(f"\n  σ_ε = {mult} × std = {eps:.6f}")

    # Step 1: Create perturbed labels with this noise level
    gen = torch.Generator().manual_seed(123)   # Same seed → same noise direction
    noise_new = torch.randn(b_tr_ours.shape, generator=gen) * eps
    b_tr_pert_new = b_tr_ours + noise_new

    # Step 2: Train perturbed FNO (same architecture, same hyperparams)
    print(f"    Training perturbed FNO...")
    model_pert_new = FNO2d(modes=12, width=32, T_in=T_in, T_out=T_out, depth=4, drop=0.1)
    model_pert_new = train_fno(model_pert_new, a_tr_ours, b_tr_pert_new,
                                a_cal, b_cal, epochs=500,
                                ckpt=f"fno_pert_eps{mult}.pt")
    model_pert_new.eval()

    # Step 3: Compute floor τ₀ for this noise level
    sigma_tr_tmp = []
    with torch.no_grad():
        for i in range(0, len(a_tr_ours), 20):
            batch = a_tr_ours[i:i+20].to(DEVICE)
            sigma_tr_tmp.append((model(batch) - model_pert_new(batch)).abs().cpu())
    sigma_tr_tmp = torch.cat(sigma_tr_tmp)
    floor_tmp = sigma_tr_tmp.median().item() * 0.1
    print(f"    Floor τ₀ = {floor_tmp:.6f}")

    # Step 4: Pre-compute scores on pool
    # We need to temporarily use model_pert_new instead of model_pert
    scores_tmp = []
    msigma_tmp = []
    with torch.no_grad():
        for i in range(0, len(pool_a), 20):
            a_batch = pool_a[i:i+20].to(DEVICE)
            b_batch = pool_b[i:i+20]

            pred = model(a_batch).cpu()
            pred_p = model_pert_new(a_batch).cpu()
            raw_diff = (pred - pred_p).abs()
            sigma = smooth_J(raw_diff, kernel_size=SMOOTH_KS_PERT).clamp(min=max(floor_tmp, 1e-12))

            err = (b_batch - pred).abs()
            ratios = err / sigma
            batch_scores = ratios.reshape(len(a_batch), -1).max(dim=1).values
            batch_mean_sig = sigma.mean(dim=[1, 2, 3])

            scores_tmp.append(batch_scores)
            msigma_tmp.append(batch_mean_sig)

    scores_arr = torch.cat(scores_tmp).numpy()
    msigma_arr = torch.cat(msigma_tmp).numpy()

    # Step 5: Run 1000 reshuffles
    print(f"    Running 1000 reshuffles...")
    cov_res, bw_res = run_reshuffle(
        scores_arr, msigma_arr, N_pool, N_CAL, N_TEST, alpha_vals, N_REPEATS=1000
    )
    noise_results[mult] = {"cov": cov_res, "bw": bw_res, "floor": floor_tmp}

    # Free GPU memory
    del model_pert_new
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# Print Noise Sensitivity Table (Table 3a in paper)
# ============================================================
print("\n" + "=" * 100)
print(f"  TABLE 3a (PAPER): NOISE SENSITIVITY (1000 reshuffles, K=15, floor=0.1×median)")
print("=" * 100)
for a_ in [0.02, 0.04, 0.06]:
    print(f"\n  α = {a_} (target coverage: {(1-a_)*100:.0f}%)")
    print(f"  {'σ_ε / std(u)':>14} {'Cov%':>8} {'±SE':>8}   {'BW':>10} {'±SE':>10}")
    print(f"  {'-'*56}")
    for mult in NOISE_MULTIPLIERS:
        covs = np.array(noise_results[mult]["cov"][a_])
        bws  = np.array(noise_results[mult]["bw"][a_])
        n = len(covs)
        marker = " ← default" if mult == 0.05 else ""
        print(f"  {mult:>14.2f} {covs.mean()*100:>7.2f}% ±{covs.std()*100/np.sqrt(n):>5.3f}%"
              f"   {bws.mean():>9.4f} ±{bws.std()/np.sqrt(n):>8.5f}{marker}")


  NOISE SENSITIVITY: Varying Perturbation Magnitude σ_ε
  σ_ε ∈ [0.01, 0.02, 0.05, 0.1, 0.2] × std(u_train)

  σ_ε = 0.01 × std = 0.014590
    Training perturbed FNO...
  ep  0  train=0.6758  val=0.4523
  ep 50  train=0.1885  val=0.2500
  ep100  train=0.1611  val=0.2489
  ep150  train=0.1493  val=0.2511
  ep200  train=0.1398  val=0.2546
  ep250  train=0.1359  val=0.2585
  ep300  train=0.1332  val=0.2616
  ep350  train=0.1307  val=0.2630
  ep400  train=0.1276  val=0.2641
  ep450  train=0.1263  val=0.2669
Best train rel-L2 = 0.1253
    Floor τ₀ = 0.006810
    Running 1000 reshuffles...

  σ_ε = 0.02 × std = 0.029179
    Training perturbed FNO...
  ep  0  train=0.6613  val=0.4343
  ep 50  train=0.1919  val=0.2527
  ep100  train=0.1650  val=0.2476
  ep150  train=0.1533  val=0.2502
  ep200  train=0.1436  val=0.2515
  ep250  train=0.1400  val=0.2536
  ep300  train=0.1373  val=0.2562
  ep350  train=0.1350  val=0.2563
  ep400  train=0.1321  val=0.2577
  ep450  train=0.1308  val=0.2588
Best tr

In [ ]:
# ============================================================
# CELL 11: RANDOM SEED SENSITIVITY
# ============================================================
#
# REVIEWER REQUEST ADDRESSED:
# "They will also want to know whether the gains survive [...]
#  multiple random seeds."
#
# We retrain both the base FNO and the perturbed FNO with 5 different
# random seeds, then evaluate each pair via 1000 reshuffles.
#
# This is the MOST EXPENSIVE experiment:
#   5 seeds × 2 models × 500 epochs ≈ 5000 training epochs total
#   Expected runtime: ~50 min (GPU) to ~5 hrs (CPU)
#
# For each seed:
#   1. Set torch.manual_seed(seed) and np.random.seed(seed)
#   2. Retrain base FNO from scratch
#   3. Retrain perturbed FNO from scratch (same perturbation noise)
#   4. Compute floor τ₀
#   5. Pre-compute scores on pool
#   6. Run 1000 reshuffles
#
# We report:
#   - Per-seed results (coverage and bandwidth for each seed)
#   - Across-seed mean ± std (the key statistic for reviewers)
# ============================================================

SEEDS = [42, 123, 256, 789, 1024]

print("\n" + "=" * 90)
print(f"  SEED SENSITIVITY ({len(SEEDS)} seeds, 1000 reshuffles each)")
print("=" * 90)

# Storage for per-seed, per-alpha results
seed_per_seed_bw  = {a: [] for a in alpha_vals}   # each entry = mean_bw for that seed
seed_per_seed_cov = {a: [] for a in alpha_vals}   # each entry = mean_cov for that seed

for seed in SEEDS:
    print(f"\n{'─'*60}")
    print(f"  Seed = {seed}")
    print(f"{'─'*60}")

    # Set random seeds for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Step 1: Retrain base FNO from scratch
    print(f"  Training base FNO (seed={seed})...")
    model_seed = FNO2d(modes=12, width=32, T_in=T_in, T_out=T_out, depth=4, drop=0.1)
    model_seed = train_fno(model_seed, a_tr_ours, b_tr_ours, a_cal, b_cal,
                           epochs=500, ckpt=f"fno_seed{seed}.pt")
    model_seed.eval()

    # Step 2: Retrain perturbed FNO from scratch
    # IMPORTANT: We keep the perturbation noise FIXED (Generator seed=123)
    # so that the only source of randomness is the network initialization/training
    EPSILON_seed = b_tr_ours.std().item() * 0.05
    gen_seed = torch.Generator().manual_seed(123)   # Same noise across seeds
    noise_seed = torch.randn(b_tr_ours.shape, generator=gen_seed) * EPSILON_seed
    b_tr_pert_seed = b_tr_ours + noise_seed

    print(f"  Training perturbed FNO (seed={seed})...")
    model_pert_seed = FNO2d(modes=12, width=32, T_in=T_in, T_out=T_out, depth=4, drop=0.1)
    model_pert_seed = train_fno(model_pert_seed, a_tr_ours, b_tr_pert_seed,
                                 a_cal, b_cal, epochs=500,
                                 ckpt=f"fno_pert_seed{seed}.pt")
    model_pert_seed.eval()

    # Step 3: Compute floor τ₀
    sigma_tr_tmp = []
    with torch.no_grad():
        for i in range(0, len(a_tr_ours), 20):
            batch = a_tr_ours[i:i+20].to(DEVICE)
            sigma_tr_tmp.append((model_seed(batch) - model_pert_seed(batch)).abs().cpu())
    sigma_tr_tmp = torch.cat(sigma_tr_tmp)
    floor_tmp = sigma_tr_tmp.median().item() * 0.1
    print(f"  Floor τ₀ = {floor_tmp:.6f}")

    # Step 4: Pre-compute scores on pool
    scores_tmp = []
    msigma_tmp = []
    with torch.no_grad():
        for i in range(0, len(pool_a), 20):
            a_batch = pool_a[i:i+20].to(DEVICE)
            b_batch = pool_b[i:i+20]

            pred = model_seed(a_batch).cpu()
            pred_p = model_pert_seed(a_batch).cpu()
            raw_diff = (pred - pred_p).abs()
            sigma = smooth_J(raw_diff, kernel_size=SMOOTH_KS_PERT).clamp(min=max(floor_tmp, 1e-12))

            err = (b_batch - pred).abs()
            ratios = err / sigma
            batch_scores = ratios.reshape(len(a_batch), -1).max(dim=1).values
            batch_mean_sig = sigma.mean(dim=[1, 2, 3])

            scores_tmp.append(batch_scores)
            msigma_tmp.append(batch_mean_sig)

    scores_arr = torch.cat(scores_tmp).numpy()
    msigma_arr = torch.cat(msigma_tmp).numpy()

    # Step 5: Run 1000 reshuffles
    print(f"  Running 1000 reshuffles...")
    cov_res, bw_res = run_reshuffle(
        scores_arr, msigma_arr, N_pool, N_CAL, N_TEST, alpha_vals, N_REPEATS=1000
    )

    # Store mean results for this seed
    for a_ in alpha_vals:
        seed_per_seed_bw[a_].append(np.mean(bw_res[a_]))
        seed_per_seed_cov[a_].append(np.mean(cov_res[a_]))

    # Report per-seed results
    print(f"\n  Seed {seed} results:")
    for a_ in [0.02, 0.04, 0.06]:
        print(f"    α={a_}: Cov={np.mean(cov_res[a_])*100:.2f}%  BW={np.mean(bw_res[a_]):.4f}")

    # Free GPU memory
    del model_seed, model_pert_seed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# Print Seed Sensitivity Table (Table 3b in paper)
# ============================================================
print("\n" + "=" * 90)
print(f"  TABLE 3b (PAPER): SEED SENSITIVITY ({len(SEEDS)} seeds × 1000 reshuffles)")
print("=" * 90)
print(f"  {'α':<8} {'Mean Cov%':>10} {'Std Cov%':>10} {'Mean BW':>10} {'Std BW':>10}")
print(f"  {'-'*52}")
for a_ in alpha_vals:
    covs = np.array(seed_per_seed_cov[a_])
    bws  = np.array(seed_per_seed_bw[a_])
    print(f"  {a_:<8.2f} {covs.mean()*100:>9.2f}% {covs.std()*100:>9.3f}%"
          f"  {bws.mean():>9.4f} {bws.std():>9.5f}")

# Also print per-seed breakdown
print(f"\n  Per-seed breakdown (α=0.04):")
print(f"  {'Seed':<8} {'Cov%':>10} {'BW':>10}")
print(f"  {'-'*30}")
a_show = 0.04
for i, seed in enumerate(SEEDS):
    print(f"  {seed:<8} {seed_per_seed_cov[a_show][i]*100:>9.2f}% "
          f"{seed_per_seed_bw[a_show][i]:>9.4f}")


  SEED SENSITIVITY (5 seeds, 1000 reshuffles each)

────────────────────────────────────────────────────────────
  Seed = 42
────────────────────────────────────────────────────────────
  Training base FNO (seed=42)...
  ep  0  train=0.6911  val=0.4605
  ep 50  train=0.1862  val=0.2463
  ep100  train=0.1609  val=0.2485
  ep150  train=0.1492  val=0.2508
  ep200  train=0.1395  val=0.2512
  ep250  train=0.1359  val=0.2554
  ep300  train=0.1333  val=0.2572
  ep350  train=0.1312  val=0.2596
  ep400  train=0.1283  val=0.2595
  ep450  train=0.1271  val=0.2606
Best train rel-L2 = 0.1262
  Training perturbed FNO (seed=42)...
  ep  0  train=0.6695  val=0.4317
  ep 50  train=0.1959  val=0.2501
  ep100  train=0.1689  val=0.2492
  ep150  train=0.1576  val=0.2491
  ep200  train=0.1490  val=0.2518
  ep250  train=0.1455  val=0.2532
  ep300  train=0.1430  val=0.2568
  ep350  train=0.1411  val=0.2558
  ep400  train=0.1385  val=0.2587
  ep450  train=0.1375  val=0.2592


In [ ]:
# ============================================================
# CELL 12: PUBLICATION FIGURES
# ============================================================
# This cell generates all figures for the paper:
#
#   Figure 1: Qualitative uncertainty map (3 panels)
#             Left: ground truth vorticity
#             Center: conformal radius Q_{1-α} × σ(a)
#             Right: coverage margin (green=covered, yellow=tight)
#
#   Figure 2: Bandwidth vs. coverage comparison (2-panel plot)
#             (a) Bandwidth vs α — how interval width changes with significance
#             (b) Bandwidth vs coverage — the fundamental efficiency trade-off
#             Both panels show ±1 SE bands from 1000 reshuffles.
#
#   Figure 3: Base operator performance check (3 panels)
#             Left: ground truth
#             Center: FNO prediction
#             Right: absolute residual
# ============================================================

import matplotlib.ticker as mticker

# ============================================================
# Figure 1: Qualitative perturbation-based uncertainty map
# ============================================================

# Use α=0.04 results. Find the (sample, time step) with the most missed pixels
# to show the most informative coverage margin.
r_pert = RES["Perturbation"][0.04]

best_idx, best_t, best_miss = 0, 0, 0
for i in range(len(a_test)):
    for t in range(T_out):
        pred_i = r_pert['pred'][i, :, :, t].numpy()
        truth_i = b_test[i, :, :, t].numpy()
        radius_i = r_pert['radius'][i, :, :, t].numpy()
        miss_pct = (np.abs(truth_i - pred_i) > radius_i).mean() * 100
        if miss_pct > best_miss:
            best_miss = miss_pct
            best_idx = i
            best_t = t

print(f"Best sample for Figure 1: idx={best_idx}, t={best_t}, miss={best_miss:.1f}%")
idx = best_idx
t_show = best_t

truth      = b_test[idx, :, :, t_show].numpy()
pred_val   = r_pert['pred'][idx, :, :, t_show].numpy()
radius_map = r_pert['radius'][idx, :, :, t_show].numpy()
error_map  = np.abs(truth - pred_val)

fig1, axes1 = plt.subplots(1, 3, figsize=(15, 4.5))

# Left: ground truth vorticity field
im0 = axes1[0].imshow(truth, cmap='jet', origin='upper')
axes1[0].set_title(f"Truth $(T={t_show + T_in})$", fontsize=13)
plt.colorbar(im0, ax=axes1[0], fraction=0.046, pad=0.04)

# Center: conformal radius Q_{1-α} × σ(a) — the prediction band half-width
im1 = axes1[1].imshow(radius_map, cmap='inferno', origin='upper')
axes1[1].set_title("Conformal radius", fontsize=13)
plt.colorbar(im1, ax=axes1[1], fraction=0.046, pad=0.04)

# Right: coverage margin = radius − |error|
# margin > 0: covered (green), margin < 0: missed (red), margin ≈ 0: barely covered (yellow)
margin = radius_map - error_map
im2 = axes1[2].imshow(margin, cmap='RdYlGn', origin='upper',
                       vmin=-np.percentile(np.abs(margin), 95),
                       vmax= np.percentile(np.abs(margin), 95))
axes1[2].set_title("Coverage margin", fontsize=13)
plt.colorbar(im2, ax=axes1[2], fraction=0.046, pad=0.04)

for ax in axes1:
    ax.set_xticks([0, 20, 40, 60])
    ax.set_yticks([0, 20, 40, 60])

plt.tight_layout()
plt.savefig("fig1_uncertainty_map.png", dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig("fig1_uncertainty_map.pdf", bbox_inches='tight', facecolor='white')
plt.show()
print("Saved fig1_uncertainty_map.png/pdf")


# ============================================================
# Figure 2: Bandwidth vs Coverage Comparison (2-panel)
# ============================================================

PLOT_METHODS = ["Perturbation", "Laplace", "MCDropout", "UQNO", "NoSigma"]

# Visual styling — our method is bold red and drawn on top
CONFIG = {
    "Perturbation": {
        "color": "#E63946", "marker": "D", "ms": 9,
        "label": "Perturbation (ours)", "zorder": 10, "lw": 2.2,
    },
    "Laplace": {
        "color": "#457B9D", "marker": "s", "ms": 7,
        "label": "Laplace", "zorder": 5, "lw": 1.5,
    },
    "MCDropout": {
        "color": "#2A9D8F", "marker": "^", "ms": 8,
        "label": "MC Dropout", "zorder": 5, "lw": 1.5,
    },
    "UQNO": {
        "color": "#E9C46A", "marker": "o", "ms": 7,
        "label": "UQNO (Ma et al.)", "zorder": 5, "lw": 1.5,
    },
    "NoSigma": {
        "color": "#888888", "marker": "x", "ms": 7,
        "label": "Unscaled (σ≡1)", "zorder": 4, "lw": 1.2,
    },
}

# Extract mean ± std for each method and alpha from 1000 reshuffles
plot_data = {}
for m in PLOT_METHODS:
    covs_mean, covs_std = [], []
    bws_mean, bws_std   = [], []
    for a_ in alpha_vals:
        c = np.array(results[m][a_]["cov"])
        b = np.array(results[m][a_]["bw"])
        covs_mean.append(np.mean(c) * 100)
        covs_std.append(np.std(c) * 100)
        bws_mean.append(np.mean(b))
        bws_std.append(np.std(b))
    plot_data[m] = {
        "cov_mean": np.array(covs_mean), "cov_std": np.array(covs_std),
        "bw_mean": np.array(bws_mean), "bw_std": np.array(bws_std),
    }

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Left panel (a): Bandwidth vs α ---
for m in PLOT_METHODS:
    d = plot_data[m]; c = CONFIG[m]
    # Shaded ±1 SE band (SE = std / √1000 ≈ small, so we show ±1 std instead for visibility)
    ax1.fill_between(alpha_vals, d["bw_mean"] - d["bw_std"], d["bw_mean"] + d["bw_std"],
                     color=c["color"], alpha=0.12)
    ax1.plot(alpha_vals, d["bw_mean"], color=c["color"], lw=c["lw"], zorder=c["zorder"])
    ax1.scatter(alpha_vals, d["bw_mean"], color=c["color"], marker=c["marker"],
                s=c["ms"]**2, zorder=c["zorder"]+1, edgecolors="white", linewidths=0.6,
                label=c["label"])

ax1.set_xlabel(r"Significance level $\alpha$", fontsize=12)
ax1.set_ylabel("Average bandwidth (lower is better)", fontsize=12)
ax1.set_yscale("log")
ax1.set_xticks(alpha_vals)
ax1.grid(True, which="major", alpha=0.15); ax1.grid(True, which="minor", alpha=0.07, ls=":")
ax1.legend(fontsize=9, framealpha=0.95, edgecolor="none", loc="upper right")
ax1.set_title("(a) Bandwidth vs significance level", fontsize=12, fontweight="bold", pad=10)

# --- Right panel (b): Bandwidth vs Coverage ---
for m in PLOT_METHODS:
    d = plot_data[m]; c = CONFIG[m]
    ax2.plot(d["cov_mean"], d["bw_mean"], color=c["color"], lw=c["lw"],
             zorder=c["zorder"], alpha=0.5)
    ax2.scatter(d["cov_mean"], d["bw_mean"], color=c["color"], marker=c["marker"],
                s=c["ms"]**2, zorder=c["zorder"]+1, edgecolors="white", linewidths=0.6,
                label=c["label"])

ax2.annotate("← better", xy=(0.02, 0.02), xycoords="axes fraction",
             fontsize=9, color="#666666", fontstyle="italic")
ax2.set_xlabel("Average coverage % (higher is better)", fontsize=12)
ax2.set_ylabel("Average bandwidth (lower is better)", fontsize=12)
ax2.set_yscale("log")
ax2.grid(True, which="major", alpha=0.15); ax2.grid(True, which="minor", alpha=0.07, ls=":")
ax2.legend(fontsize=9, framealpha=0.95, edgecolor="none", loc="upper left")
ax2.set_title("(b) Bandwidth vs coverage", fontsize=12, fontweight="bold", pad=10)

fig.suptitle(
    f"2D Navier-Stokes ($\\nu=10^{{-5}}$): Conformal Prediction Band Comparison\n"
    f"$N_{{\\mathrm{{train}}}}$={N_TR_OURS}, "
    f"$N_{{\\mathrm{{cal}}}}$={N_CAL}, "
    f"$N_{{\\mathrm{{test}}}}$={N_TEST}, "
    f"{N_REPEATS} reshuffles",
    fontsize=13, fontweight="bold", y=1.04
)

plt.tight_layout()
plt.savefig("fig2_bw_comparison.png", dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig("fig2_bw_comparison.pdf", bbox_inches='tight', facecolor='white')
plt.show()
print("Saved fig2_bw_comparison.png/pdf")


# ============================================================
# Figure 3: Base operator performance check
# ============================================================
# Verify that the base FNO is a reasonable surrogate before
# the conformal wrapper is applied.

r_pert = RES["Perturbation"][0.04]
fig3, axes3 = plt.subplots(1, 3, figsize=(15, 4.5))

# Pick the last time step for variety
t_check = T_out - 1
idx_check = 0

truth_check = b_test[idx_check, :, :, t_check].numpy()
pred_check  = r_pert['pred'][idx_check, :, :, t_check].numpy()
err_check   = np.abs(truth_check - pred_check)

im0 = axes3[0].imshow(truth_check, cmap='jet', origin='upper')
axes3[0].set_title(f"Truth (T={t_check + T_in})", fontsize=13)
plt.colorbar(im0, ax=axes3[0], fraction=0.046, pad=0.04)

im1 = axes3[1].imshow(pred_check, cmap='jet', origin='upper')
axes3[1].set_title(f"Prediction (T={t_check + T_in})", fontsize=13)
plt.colorbar(im1, ax=axes3[1], fraction=0.046, pad=0.04)

im2 = axes3[2].imshow(err_check, cmap='hot', origin='upper')
axes3[2].set_title("Error", fontsize=13)
plt.colorbar(im2, ax=axes3[2], fraction=0.046, pad=0.04)

for ax in axes3:
    ax.set_xticks([0, 20, 40, 60])
    ax.set_yticks([0, 20, 40, 60])

plt.tight_layout()
plt.savefig("fig3_base_operator.png", dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig("fig3_base_operator.pdf", bbox_inches='tight', facecolor='white')
plt.show()
print("Saved fig3_base_operator.png/pdf")

print("\n" + "=" * 60)
print("  ALL EXPERIMENTS COMPLETE")
print("=" * 60)